# nbk_covid_SZ_to_GZ
## Silver → Gold Transformation

**Purpose:** Read clean COVID-19 data from the Silver layer, aggregate and enrich it, and write analytical outputs to the Gold layer.

**Input:** `Silver/covid_worldwide_SZ.csv`

**Outputs (Gold layer):**
- `covid_worldwide_GZ.csv` — Full enriched dataset with calculated metrics
- `top_cases_GZ.csv` — Top 10 countries by total deaths
- `less_cases_GZ.csv` — Countries with lowest death counts
- `percentage_of_death_GZ.csv` — Countries ranked by death rate
- `best_recovery_GZ.csv` — Countries ranked by recovery gap

### Aggregations applied
- `percentage of patients death` = Total Deaths / Total Cases
- `recovered by cases` = Total Cases - Total Recovered
- Rankings by death count, death rate, and recovery gap

> ⚠️ **Before running:** update the ABFSS paths to match your own storage account name.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [ ]:
# ⚠️ Update storage account name if different
covid_proc = pd.read_csv(
    'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Silver/covid_worldwide_SZ.csv',
    index_col=False
)
covid_proc.head()

In [ ]:
covid_proc.info()

In [ ]:
# Metric 1: Death rate per country
covid_proc['percentage of patients death'] = covid_proc['Total Deaths'] / covid_proc['Total Cases']
covid_proc[['Country', 'percentage of patients death']].head()

In [ ]:
# Metric 2: Top 10 countries with highest death count
top_cases = covid_proc[['Country', 'Total Deaths']].sort_values(by='Total Deaths', ascending=False)[:10]
top_cases

In [ ]:
# Metric 3: Countries with lowest death count
less_cases = covid_proc[['Country', 'Total Deaths']].sort_values(by='Total Deaths', ascending=False)[187:]
less_cases

In [ ]:
# Metric 4: Recovery gap (Total Cases - Total Recovered) — lower = better recovery
covid_proc['recovered by cases'] = covid_proc['Total Cases'] - covid_proc['Total Recovered']
best_recovery = covid_proc[['Country', 'recovered by cases']].sort_values(by='recovered by cases', ascending=False)
best_recovery[:10]

In [ ]:
# Metric 5: Top 10 countries by death rate percentage
percentage_of_death = covid_proc[['Country', 'percentage of patients death']].sort_values(
    by='percentage of patients death', ascending=False
)
percentage_of_death[:10]

In [ ]:
# Export: Full enriched dataset → Gold
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Gold/covid_worldwide_GZ.csv'
covid_proc.to_csv(csv_file_path, index=False)
print(f'Exported: {csv_file_path}')

In [ ]:
# Export: Top 10 countries by death count → Gold
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Gold/top_cases_GZ.csv'
top_cases.to_csv(csv_file_path, index=False)
print(f'Exported: {csv_file_path}')

In [ ]:
# Export: Countries with lowest death count → Gold
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Gold/less_cases_GZ.csv'
less_cases.to_csv(csv_file_path, index=False)
print(f'Exported: {csv_file_path}')

In [ ]:
# Export: Death rate per country → Gold
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Gold/percentage_of_death_GZ.csv'
percentage_of_death.to_csv(csv_file_path, index=False)
print(f'Exported: {csv_file_path}')

In [ ]:
# Export: Recovery gap ranking → Gold
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Gold/best_recovery_GZ.csv'
best_recovery.to_csv(csv_file_path, index=False)
print(f'Exported: {csv_file_path}')